# Experiment 15.3 — FT-Transformer Multi-Dataset Validation

**Objective:** Prove that the FYDP-III FT-Transformer pipeline (Interpretability via Attention & Ablation) generalizes beyond the original cHL CODEX dataset.

| Dataset | Disease | Imaging | Cells | Markers | Cell Types |
|---------|---------|---------|-------|---------|------------|
| cHL CODEX (Original) | Hodgkin Lymphoma | CODEX | 145K | 49 | 16 |
| CRC CODEX | Colorectal Cancer | CODEX | 258K | 56 | ~25 |
| cHL MIBI | Hodgkin Lymphoma | MIBI | 1.67M | 41 | 14 |


In [1]:
import os, random, time, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

warnings.filterwarnings('ignore', category=UserWarning)

SEED = 7325111

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')


PyTorch: 2.10.0+cu128
Device: cuda


## FT-Transformer Infrastructure (From Experiment 15.1)


In [2]:
class CellDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def make_loaders(X_tr, y_tr, X_va, y_va, batch_size=256):
    ds_tr = CellDataset(X_tr, y_tr)
    ds_va = CellDataset(X_va, y_va)

    counts   = np.bincount(y_tr)
    weights  = 1.0 / counts[y_tr]
    sampler  = WeightedRandomSampler(weights, len(weights), replacement=True)

    tr_loader = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=True)
    va_loader = DataLoader(ds_va, batch_size=batch_size, sampler=SequentialSampler(ds_va), drop_last=False)
    return tr_loader, va_loader

class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_token))
        self.b = nn.Parameter(torch.zeros(n_features, d_token))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.W + self.b

class FTTransformer(nn.Module):
    def __init__(self, n_features=50, d_token=64, n_heads=8, n_layers=3, ffn_mult=4, dropout=0.1, n_classes=16):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token*ffn_mult,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.Linear(d_token, n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)
        tokens = self.tokenizer(x)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.encoder(tokens)
        return self.head(out[:, 0])

    @torch.no_grad()
    def get_attention_maps(self, x: torch.Tensor):
        B = x.size(0)
        tokens = self.tokenizer(x)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        attn_maps = []
        for layer in self.encoder.layers:
            src = layer.norm1(tokens) if layer.norm_first else tokens
            _, w = layer.self_attn(src, src, src, need_weights=True, average_attn_weights=False)
            attn_maps.append(w.cpu())
            tokens = layer(tokens)
        return attn_maps

def train_ft_transformer(X_tr, y_tr, X_va, y_va, n_features, n_classes, d_token=64, n_heads=8, n_layers=3, max_epochs=200, patience=50, min_epochs=100):
    set_seed(SEED)
    tr_loader, va_loader = make_loaders(X_tr, y_tr, X_va, y_va, batch_size=256)
    model = FTTransformer(n_features=n_features, n_classes=n_classes, d_token=d_token, n_heads=n_heads, n_layers=n_layers).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion = nn.CrossEntropyLoss()

    best_loss, best_state, patience_ctr = float('inf'), None, 0

    print(f'Training FT-Transformer... Params: {sum(p.numel() for p in model.parameters()):,}')
    for epoch in range(max_epochs):
        model.train()
        for X_b, y_b in tr_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b.to(DEVICE)), y_b.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in va_loader:
                logits = model(X_b.to(DEVICE))
                va_loss += criterion(logits, y_b.to(DEVICE)).item() * len(y_b)
        va_loss /= len(y_va)

        if va_loss < best_loss:
            best_loss = val_loss = va_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            if epoch % 20 == 0: print(f'Epoch {epoch:3d} | va_loss={va_loss:.4f} ← best')
        else:
            patience_ctr += 1

        if patience_ctr > patience and epoch >= min_epochs:
            print(f'Early stop at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, X_sc):
    model.eval()
    ds = CellDataset(X_sc, np.zeros(len(X_sc)))
    loader = DataLoader(ds, batch_size=512, shuffle=False)
    probs = [torch.softmax(model(X_b.to(DEVICE)), dim=-1).cpu().numpy() for X_b, _ in loader]
    return np.concatenate(probs, axis=0)


## Reusable Pipeline Function


In [3]:
def run_ft_transformer_pipeline(X, y, feature_cols, class_names, dataset_name, max_cells=None):
    """Runs the full FT-Transformer pipeline + Ablation Study on any dataset."""
    set_seed(SEED)
    NUM_FEATURES = X.shape[1]
    NUM_CLASSES = len(class_names)

    if max_cells and len(X) > max_cells:
        print(f"Subsampling {len(X)} -> {max_cells} cells (stratified)...")
        idx = np.arange(len(X))
        idx, _ = train_test_split(idx, train_size=max_cells, random_state=SEED, stratify=y)
        X, y = X[idx], y[idx]

    print(f'\n{"="*70}')
    print(f'RUNNING FT-TRANSFORMER PIPELINE ON: {dataset_name}')
    print(f'{"="*70}')
    print(f'Cells: {len(X):,} | Features: {NUM_FEATURES} | Classes: {NUM_CLASSES}')

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y)

    # Standard Scaling (Crucial for deep learning on tabular data)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_valid_sc = scaler.transform(X_valid)

    # 1. Train FT-Transformer
    print("\n1. Training FT-Transformer...")
    ft_model = train_ft_transformer(X_train_sc, y_train, X_valid_sc, y_valid,
                                    n_features=NUM_FEATURES, n_classes=NUM_CLASSES)

    ft_probs = predict(ft_model, X_valid_sc)
    ft_preds = np.argmax(ft_probs, axis=1)
    ft_f1 = f1_score(y_valid, ft_preds, average='weighted')
    print(f'\n   FT-Transformer F1: {ft_f1:.4f}')

    # 2. Extract Attention & Run Ablation Study
    print("\n2. Running Causal Ablation Study...")
    ft_model.eval()

    # Get global attention (average over all validation samples)
    print('   Extracting attention maps...')
    ds_val_tensor = torch.tensor(X_valid_sc, dtype=torch.float32)
    batch_size = 512
    all_attn = []
    with torch.no_grad():
        for i in range(0, len(ds_val_tensor), batch_size):
            batch = ds_val_tensor[i:i+batch_size].to(DEVICE)
            # get_attention_maps returns list of layers, we want the last layer
            attn_layers = ft_model.get_attention_maps(batch)
            last_layer_attn = attn_layers[-1] # Shape: (B, num_heads, num_tokens, num_tokens)
            # Average across heads. Token 0 is CLS token. We want CLS attention to features (1 to N)
            cls_attn = last_layer_attn.mean(dim=1)[:, 0, 1:] # Shape: (B, num_features)
            all_attn.append(cls_attn.cpu().numpy())
    
    global_attn = np.concatenate(all_attn, axis=0)

    # Aggregate attention per cell type
    class_attention = {}
    for c in range(NUM_CLASSES):
        mask = (y_valid == c)
        if mask.sum() > 0:
            class_attention[c] = global_attn[mask].mean(axis=0)
        else:
            class_attention[c] = np.zeros(NUM_FEATURES)

    # Base Recalls
    from sklearn.metrics import recall_score
    base_recalls = recall_score(y_valid, ft_preds, average=None)

    ablation_results = []
    print('   Masking Top-3 vs Random-3 features per cell type...')
    for c in range(NUM_CLASSES):
        c_name = class_names[c]
        mask = (y_valid == c)
        if mask.sum() == 0: continue

        attn_scores = class_attention[c]
        top3_idx = np.argsort(attn_scores)[-3:]
        
        # Get random 3 features (excluding the top 3)
        available_idx = list(set(range(NUM_FEATURES)) - set(top3_idx))
        random3_idx = np.random.choice(available_idx, 3, replace=False)

        # Mask Top 3
        X_abl_top = X_valid_sc.copy()
        X_abl_top[:, top3_idx] = 0.0
        preds_top = np.argmax(predict(ft_model, X_abl_top), axis=1)
        top_recall = recall_score(y_valid[mask], preds_top[mask], average='micro')

        # Mask Random 3
        X_abl_rand = X_valid_sc.copy()
        X_abl_rand[:, random3_idx] = 0.0
        preds_rand = np.argmax(predict(ft_model, X_abl_rand), axis=1)
        rand_recall = recall_score(y_valid[mask], preds_rand[mask], average='micro')

        ablation_results.append({
            'Cell Type': c_name,
            'Base Recall': base_recalls[c],
            'Mask Top-3 Recall': top_recall,
            'Mask Random-3 Recall': rand_recall,
            'Drop (Top-3)': base_recalls[c] - top_recall,
            'Drop (Random-3)': base_recalls[c] - rand_recall
        })

    df_abl = pd.DataFrame(ablation_results)
    avg_drop_top = df_abl['Drop (Top-3)'].mean()
    avg_drop_rand = df_abl['Drop (Random-3)'].mean()

    print(f'\n   Average Recall Drop (Top-3): {-avg_drop_top:.4f}')
    print(f'   Average Recall Drop (Random-3): {-avg_drop_rand:.4f}')
    if avg_drop_top > avg_drop_rand:
        print('   ✅ Causal link confirmed: Masking attention-selected markers causes a larger drop.')

    # Print Results
    print(f'\n{"="*70}')
    print(f'RESULTS: {dataset_name}')
    print(f'{"="*70}')
    print(f'  FT-Transformer W-F1: {ft_f1:.4f}')
    print(f'  Avg Drop (Top-3): {-avg_drop_top:.4f}')
    print(f'  Avg Drop (Rand-3): {-avg_drop_rand:.4f}')

    return {'dataset': dataset_name, 'ft_f1': ft_f1, 
            'avg_drop_top': -avg_drop_top, 'avg_drop_rand': -avg_drop_rand,
            'num_cells': len(X), 'num_features': NUM_FEATURES, 'num_classes': NUM_CLASSES}


---
## Dataset 1: CRC CODEX (Colorectal Cancer)
Same imaging technology (CODEX), different disease → proves **cross-tissue generalization**.


In [4]:
print("Loading CRC CODEX dataset...")
df_crc = pd.read_csv("/kaggle/input/datasets/imranbhuiyan999/crc-clusters-neighborhoods-markers/CRC_clusters_neighborhoods_markers.csv")
print(f"Raw: {len(df_crc)} cells")

# Drop noisy / ambiguous classes
drop_classes = ['dirt', 'undefined', 'immune cells / vasculature', 'tumor cells / immune cells']
df_crc = df_crc[~df_crc['ClusterName'].isin(drop_classes)]
print(f"After cleanup: {len(df_crc)} cells, {df_crc['ClusterName'].nunique()} cell types")

# Identify marker columns (contain ':Cyc_')
meta_cols = ['Unnamed: 0','CellID','ClusterID','EventID','File Name','Region',
    'TMA_AB','TMA_12','Index in File','groups','patients','spots',
    'cell_id:cell_id','tile_nr:tile_nr','X:X','Y:Y',
    'X_withinTile:X_withinTile','Y_withinTile:Y_withinTile','Z:Z',
    'size:size','HOECHST1:Cyc_1_ch_1','DRAQ5:Cyc_23_ch_4',
    'Profile_Homogeneity:Fiter1','ClusterSize','ClusterName',
    'neighborhood10','neighborhood number final','neighborhood name']
binary_cols = [c for c in df_crc.columns if '+' in c]
exclude = set(meta_cols + binary_cols)
marker_cols_crc = [c for c in df_crc.columns if c not in exclude]
marker_names_crc = [c.split(' - ')[0].split(':')[0].strip() for c in marker_cols_crc]
print(f"Markers: {len(marker_cols_crc)}")

le_crc = LabelEncoder()
y_crc = le_crc.fit_transform(df_crc['ClusterName'].values)
class_names_crc = le_crc.classes_.tolist()
X_crc = df_crc[marker_cols_crc].values.astype(np.float32)
print(f"Ready: X={X_crc.shape}, Classes={len(class_names_crc)}")


Loading CRC CODEX dataset...
Raw: 258385 cells
After cleanup: 240554 cells, 25 cell types
Markers: 56
Ready: X=(240554, 56), Classes=25


### Run Pipeline on CRC CODEX


In [5]:
crc_results = run_ft_transformer_pipeline(
    X_crc, y_crc, marker_names_crc, class_names_crc,
    dataset_name="CRC CODEX (Colorectal Cancer)")



RUNNING FT-TRANSFORMER PIPELINE ON: CRC CODEX (Colorectal Cancer)
Cells: 240,554 | Features: 56 | Classes: 25

1. Training FT-Transformer...
Training FT-Transformer... Params: 158,937
Epoch   0 | va_loss=0.9506 ← best
Early stop at epoch 100

   FT-Transformer F1: 0.8590

2. Running Causal Ablation Study...
   Extracting attention maps...
   Masking Top-3 vs Random-3 features per cell type...

   Average Recall Drop (Top-3): -0.5136
   Average Recall Drop (Random-3): -0.1122
   ✅ Causal link confirmed: Masking attention-selected markers causes a larger drop.

RESULTS: CRC CODEX (Colorectal Cancer)
  FT-Transformer W-F1: 0.8590
  Avg Drop (Top-3): -0.5136
  Avg Drop (Rand-3): -0.1122


---
## Dataset 2: cHL MIBI (Hodgkin Lymphoma — MIBI)
Same disease, different imaging technology → proves **cross-platform generalization**.


In [6]:
print("Loading cHL MIBI dataset...")
df_mibi = pd.read_csv("/kaggle/input/datasets/imranbhuiyan999/chl-1-mibi/cHL1_MIBI.csv")
print(f"Raw: {len(df_mibi)} cells")

mibi_meta = ['cellLabel','Annotation','centroidX','centroidY','cellSize','identifier']
marker_cols_mibi = [c for c in df_mibi.columns if c not in mibi_meta]
marker_names_mibi = marker_cols_mibi.copy()
print(f"Markers: {len(marker_cols_mibi)}, Cell types: {df_mibi['Annotation'].nunique()}")

le_mibi = LabelEncoder()
y_mibi = le_mibi.fit_transform(df_mibi['Annotation'].values)
class_names_mibi = le_mibi.classes_.tolist()
X_mibi = df_mibi[marker_cols_mibi].values.astype(np.float32)
print(f"Ready: X={X_mibi.shape}, Classes={len(class_names_mibi)}")


Loading cHL MIBI dataset...
Raw: 1669853 cells
Markers: 41, Cell types: 14
Ready: X=(1669853, 41), Classes=14


### Run Pipeline on cHL MIBI
> Subsampling to 200K cells for GPU memory.


In [7]:
mibi_results = run_ft_transformer_pipeline(
    X_mibi, y_mibi, marker_names_mibi, class_names_mibi,
    dataset_name="cHL MIBI (Hodgkin Lymphoma)", max_cells=200000)


Subsampling 1669853 -> 200000 cells (stratified)...

RUNNING FT-TRANSFORMER PIPELINE ON: cHL MIBI (Hodgkin Lymphoma)
Cells: 200,000 | Features: 41 | Classes: 14

1. Training FT-Transformer...
Training FT-Transformer... Params: 156,302
Epoch   0 | va_loss=0.7043 ← best
Epoch  20 | va_loss=0.3592 ← best
Epoch  40 | va_loss=0.3067 ← best
Early stop at epoch 112

   FT-Transformer F1: 0.8982

2. Running Causal Ablation Study...
   Extracting attention maps...
   Masking Top-3 vs Random-3 features per cell type...

   Average Recall Drop (Top-3): -0.4216
   Average Recall Drop (Random-3): -0.0885
   ✅ Causal link confirmed: Masking attention-selected markers causes a larger drop.

RESULTS: cHL MIBI (Hodgkin Lymphoma)
  FT-Transformer W-F1: 0.8982
  Avg Drop (Top-3): -0.4216
  Avg Drop (Rand-3): -0.0885


---
## Cross-Dataset Comparison Table


In [8]:
orig = {'dataset':'cHL CODEX (Original)', 'ft_f1':0.8715,
    'num_cells':145161, 'num_features':50, 'num_classes':16,
    'avg_drop_top': -0.603, 'avg_drop_rand': -0.085}

all_r = [orig, crc_results, mibi_results]

print('\n' + '='*90)
print('CROSS-DATASET GENERALIZATION RESULTS: FT-TRANSFORMER & INTERPRETABILITY')
print('='*90)
hdr = f'{"Metric":<25}'
for r in all_r: hdr += f' | {r["dataset"][:22]:>22}'
print(hdr)
print('-'*90)

for label, key, fmt in [('Cells','num_cells','{:,.0f}'),('Features','num_features','{:d}'),
    ('Cell Types','num_classes','{:d}'),('FT-Transformer F1','ft_f1','{:.4f}'),
    ('Ablation Drop (Top-3)','avg_drop_top','{:.4f}'),
    ('Ablation Drop (Rand-3)','avg_drop_rand','{:.4f}')]:
    row = f'  {label:<23}'
    for r in all_r: row += f' | {fmt.format(r[key]):>22}'
    print(row)

print(f'\n{"="*90}')
print('KEY FINDINGS:')
causal_wins = all(abs(r['avg_drop_top']) > abs(r['avg_drop_rand']) for r in all_r)
if causal_wins: 
    print('\n  ✅ CONFIRMED: Masking the Top-3 attention markers consistently causes a drastically larger drop')
    print('               than masking 3 random markers across all datasets. This proves that the')
    print('               Self-Attention mechanism is causally discovering the true biological markers.')
print('='*90)



CROSS-DATASET GENERALIZATION RESULTS: FT-TRANSFORMER & INTERPRETABILITY
Metric                    |   cHL CODEX (Original) | CRC CODEX (Colorectal  | cHL MIBI (Hodgkin Lymp
------------------------------------------------------------------------------------------
  Cells                   |                145,161 |                240,554 |                200,000
  Features                |                     50 |                     56 |                     41
  Cell Types              |                     16 |                     25 |                     14
  FT-Transformer F1       |                 0.8715 |                 0.8590 |                 0.8982
  Ablation Drop (Top-3)   |                -0.6030 |                -0.5136 |                -0.4216
  Ablation Drop (Rand-3)  |                -0.0850 |                -0.1122 |                -0.0885

KEY FINDINGS:

  ✅ CONFIRMED: Masking the Top-3 attention markers consistently causes a drastically larger drop
               t